# Data Loading

This notebook downloads, extracts, and provides an initial inspection of the FAR-Trans dataset.
It covers all seven data files: transactions, customer information, asset information, markets, close prices, limit prices, and questionnaires.

In [1]:
from pathlib import Path
import zipfile
import urllib.request
import shutil
import pandas as pd


REQUIRED_FILES = (
    "transactions.csv",
    "customer_information.csv",
    "asset_information.csv",
    "markets.csv",
    "close_prices.csv",
    "limit_prices.csv",
    "questionnaires.csv",
)


def download_and_prepare_far_trans(root: Path) -> None:
    """Download, extract, and flatten the FAR-Trans dataset into `root`.

    Idempotent: if every required CSV is already present at `root`, this is a no-op.
    """
    root.mkdir(parents=True, exist_ok=True)

    if all((root / filename).is_file() for filename in REQUIRED_FILES):
        return

    zip_url = "https://researchdata.gla.ac.uk/1658/1/FAR-Trans.zip"
    zip_path = root / "FAR-Trans.zip"

    if not zip_path.exists():
        urllib.request.urlretrieve(zip_url, zip_path)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(root)

    nested_folder = root / "FAR-Trans"

    if nested_folder.exists() and nested_folder.is_dir():
        for item in nested_folder.iterdir():
            destination = root / item.name

            if destination.exists():
                if destination.is_file():
                    destination.unlink()
                else:
                    shutil.rmtree(destination)

            shutil.move(str(item), root)

        shutil.rmtree(nested_folder)

    zip_path.unlink(missing_ok=True)


def find_file(root: Path, filename: str) -> Path:
    """Locate `filename` directly under `root`, or recursively as a fallback."""
    direct_match = root / filename
    if direct_match.is_file():
        return direct_match

    matches = list(root.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Couldn't find {filename} under {root}")

    return matches[0]


def load_far_trans_dataset(root: Path) -> dict[str, pd.DataFrame | str]:
    """Load all FAR-Trans dataset files into pandas objects (plus questionnaire text)."""
    paths = {fn: find_file(root, fn) for fn in REQUIRED_FILES}

    return {
        "transactions": pd.read_csv(paths["transactions.csv"], parse_dates=["timestamp"]),
        "customers": pd.read_csv(
            paths["customer_information.csv"],
            parse_dates=["lastQuestionnaireDate", "timestamp"],
        ),
        "assets": pd.read_csv(paths["asset_information.csv"], parse_dates=["timestamp"]),
        "markets": pd.read_csv(paths["markets.csv"]),
        "close_prices": pd.read_csv(paths["close_prices.csv"], parse_dates=["timestamp"]),
        "limit_prices": pd.read_csv(
            paths["limit_prices.csv"], parse_dates=["minDate", "maxDate"]
        ),
        "questionnaire": paths["questionnaires.csv"].read_text(
            encoding="utf-8", errors="replace"
        ),
    }

In [2]:
ROOT = Path("../data/original")

download_and_prepare_far_trans(ROOT)

dataset = load_far_trans_dataset(ROOT)

transactions = dataset["transactions"]
customers = dataset["customers"]
assets = dataset["assets"]
markets = dataset["markets"]
close_prices = dataset["close_prices"]
limit_prices = dataset["limit_prices"]
questionnaire = dataset["questionnaire"]

for name in ["transactions", "customers", "assets", "markets", "close_prices", "limit_prices"]:
    frame = dataset[name]
    print(f"{name:14s}: {len(frame):>8,} rows, {frame.shape[1]} cols")

transactions  :  388,048 rows, 9 cols
customers     :   32,468 rows, 6 cols
assets        :      836 rows, 9 cols
markets       :       38 rows, 8 cols
close_prices  :  703,303 rows, 3 cols
limit_prices  :      807 rows, 6 cols


In [3]:
customers.head(3)

,customerID,customerType,riskLevel,investmentCapacity,lastQuestionnaireDate,timestamp
0,DED5BF19E23CCCFEE322,Premium,Balanced,CAP_80K_300K,2021-11-30,2021-03-19
1,DED5BF19E23CCCFEE322,Premium,Balanced,CAP_80K_300K,2021-11-30,2022-01-21
2,6C0C752E66D5F0486C71,Mass,Income,Predicted_CAP_LT30K,2015-04-27,2018-01-02


In [4]:
transactions.head(3)

,customerID,ISIN,transactionID,transactionType,timestamp,totalValue,units,channel,marketID
0,00017496858921195E5A,GRS434003000,7590224,Buy,2020-03-27,11000.0,5000.0,Internet Banking,XATH
1,00017496858921195E5A,GRS434003000,7607029,Sell,2020-04-06,12080.0,5000.0,Internet Banking,XATH
2,00017496858921195E5A,GRS434003000,7634872,Buy,2020-04-24,13400.0,5000.0,Internet Banking,XATH


In [5]:
assets.head(3)

,ISIN,assetName,assetShortName,assetCategory,assetSubCategory,marketID,sector,industry,timestamp
0,GRF000011004,DHLOS PET OTE,DELPOIB,MTF,Balanced,AEDAK,NaN,NaN,2018-01-02
1,GRF000014008,DELOS FIXED INCOME PLUS - BOND FUND,DELDBDF,MTF,Structured,AEDAK,NaN,NaN,2018-01-02
2,GRF000022001,ΔΗΛΟΣ ΒΡΑΧΥΠΡΟΘΕΣΜΩΝ & ΜΕΣΟΠΡΟΘΕΣΜΩΝ ΕΠΕΝΔΥΣΕΩ...,DELPIMM,MTF,Money Market,AEDAK,NaN,NaN,2018-01-02


In [6]:
markets.head(3)

,exchangeID,marketID,name,description,country,tradingDays,tradingHours,marketClass
0,AEDAK,AEDAK,AEDAK Funds,Mutual funds offered by EFG Eurobank and Deuts...,Greece,"Mon,Tue,Wed,Thu,Fri",08:15-15:20,Public Securities
1,XAMS,ALXA,Euronext - Growth Amsterdam,The Euronext is a pan-European bourse providin...,Netherlands,"Mon,Tue,Wed,Thu,Fri",08:00-16:30,Public Securities
2,XBRU,ALXB,Euronext - Growth Brussels,The Euronext is a pan-European bourse providin...,Belgium,"Mon,Tue,Wed,Thu,Fri",08:00-16:30,Public Securities


In [7]:
limit_prices.head(3)

,ISIN,minDate,maxDate,priceMinDate,priceMaxDate,profitability
0,100974271034,2018-01-02,2022-11-29,3.330,4.165,0.250751
1,BE0974271034,2018-01-02,2022-11-29,3.345,4.140,0.237668
2,BE0974293251,2018-01-02,2022-11-29,93.260,56.200,-0.397384


In [8]:
close_prices.head(3)

,ISIN,timestamp,closePrice
0,100974271034,2018-01-02,3.33
1,100974271034,2018-01-03,3.33
2,100974271034,2018-01-04,3.43


In [9]:
print(questionnaire[:500])


Demographics

	1. What age group do you belong to?
		a. Younger than 35 years old
		b. Between 36 and 50
		c. Between 51 and 65
		d. Between 66 and 80
		e. Older than 80 years old

	2. Gender:
		a. Woman
		b. Man

	3. You are:
		a. Single
		b. Married
		c. Divorced
		d. Widower

	4. Regarding your studies, you have completed:
		a. Elementary Education
		b. Secondary Education
		c. Studies in a Higher Educational Institution / Higher Technological Educational Institution/ Other educational insti
